# Agent State, Memory, and Quality Gates

**Docker image**: `ml4t`

The agent in [`01_react_reasoning`](01_react_reasoning.ipynb) kept everything it knew in its
message history. That is enough to produce an answer and not enough to defend one: when the
conversation ends there is no record of what evidence was considered, no way to re-run the
analysis from where it went wrong, and no way to compare two runs except by reading two
transcripts. This notebook replaces the transcript with a typed record the agent writes to,
adds checks that refuse to let a thin run produce a confident answer, and shows what that
record makes possible once it exists.

**Learning Objectives**:
- Write the state a research run depends on into a typed record, separate from what the model
  can see
- Refuse to answer when the evidence is too thin, too old, or dated after the question's
  cutoff, and record which check refused
- Save a run to JSON mid-flight and restore it, so a run can be resumed rather than restarted
- Re-run a saved state with one class of evidence removed, to find out whether that evidence
  was doing any work

**Book Reference**: Chapter 24, Section 24.3 (Agent Memory: State, Persistence,
and Replay)

**Prerequisites**: [`01_react_reasoning`](01_react_reasoning.ipynb) (providers),
[`02_tool_contracts`](02_tool_contracts.ipynb) (tools).

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 代理狀態、記憶與品質門檻

**Docker 映像檔**：`ml4t`

[`01_react_reasoning`](01_react_reasoning.ipynb) 中的代理將所有已知資訊保存在訊息歷史中。這足以產生答案，卻不足以為答案辯護：對話結束後，沒有紀錄顯示考慮過哪些證據，無法從分析出錯之處重新執行，也只能閱讀兩份逐字稿來比較兩次執行。本 Notebook 以代理會寫入的具型別紀錄取代逐字稿，加入檢查以防證據不足的執行產生自信的答案，並展示這份紀錄建立後能做到什麼。

**學習目標**：
- 將研究執行所需的狀態寫入具型別紀錄，並與模型能看到的內容分開
- 當證據過少、過舊或日期晚於問題的截止日，就拒絕作答，並記錄是哪項檢查拒絕
- 在執行途中將狀態存成 JSON，再還原狀態，讓執行能從中途繼續，而非從頭開始
- 移除某類證據後重新執行已儲存的狀態，以判斷這類證據是否實際影響結果

**書籍參考**：第 24 章，第 24.3 節（代理記憶：狀態、持久化與重播）

**先備知識**：[`01_react_reasoning`](01_react_reasoning.ipynb)（供應者）、[`02_tool_contracts`](02_tool_contracts.ipynb)（工具）。
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""Agent State, Memory, and Quality Gates: explicit, inspectable run state."""

import json
from datetime import date, datetime, timedelta

from agent_fixtures import get_demo_question
from agent_schemas import (
    AgentState,
    check_consistency_gate,
    check_coverage_gate,
    check_freshness_gate,
    run_quality_gates,
)

## Settings

`AS_OF_ISO` is the moment the run is pretending to happen. Everything time-dependent is
measured against it rather than against the clock, which is what lets this notebook produce
the same output today and next year. It is set to the day before the demonstration
question's 2025-02-20 cutoff, so the run sits where a forecaster answering that question
would have sat.

`RUN_ID` names the run. A checkpoint is only useful if it can be told apart from the next
one, and a fixed value here keeps the printed output stable; a real deployment generates one
per run, which is what `AgentState` does when none is given.

<!-- ML4T_ZH_BEGIN c0002-p000 -->
## 設定

`AS_OF_ISO` 是這次執行假設發生的時間點。所有與時間有關的衡量都以它為準，而非以目前時鐘為準，因此本 Notebook 在今天和明年都能產生相同輸出。它設定為示範問題截止日 2025-02-20 的前一天，讓這次執行處於當時預測者回答該問題時的位置。

`RUN_ID` 用來識別這次執行。檢查點只有在能與下一次執行區分時才有用；此處使用固定值，讓印出的輸出保持穩定。實際部署時，每次執行都會產生一個識別值；未提供識別值時，`AgentState` 就會這麼做。
<!-- ML4T_ZH_END c0002-p000 -->


In [2]:
AS_OF_ISO = "2025-02-19T12:00:00"
RUN_ID = "ch24-state-demo"

In [3]:
as_of = datetime.fromisoformat(AS_OF_ISO)

## Why a Trace Is Not State

[`01_react_reasoning`](01_react_reasoning.ipynb) already writes a durable record: `RunTrace`
saves the question, every prompt and reply, and every search result into a JSON file, and
that file is what the replay path reads back. So the run is reproducible, and a reader can
see exactly what happened.

What that record cannot do is participate in the run. It is written at the end, from the
outside, and holds the model's conversation rather than the agent's own conclusions. Four
things a research process needs are all still missing:

- **A place for derived knowledge.** The evidence the agent judged relevant, the questions it
  has not resolved, what it has decided so far. None of that is a prompt or a reply.
- **Something to check before answering.** A gate has to read a structured account of the
  evidence, not a transcript, and it has to run while the agent can still act on it.
- **A resumption point.** A conversation log replays a run; it does not let one continue from
  the middle with a changed setting.
- **A comparable object.** Two runs are two conversations, and no diff over prose is a
  measurement. Two `AgentState` records diff field by field.

So the trace and the state are different artifacts with different jobs, and this chapter
keeps both: `RunTrace` is what the run looked like from outside, `AgentState` is what the
agent knew from inside.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
## 為何軌跡不等於狀態

[`01_react_reasoning`](01_react_reasoning.ipynb) 已經會寫入持久紀錄：`RunTrace` 將問題、每則提示詞與回覆，以及每項搜尋結果存進 JSON 檔案；重播路徑會讀取該檔案。因此執行過程可重現，讀者也能確切看到發生了什麼。

但這份紀錄無法參與執行過程。它是在結束時從外部寫入，保存的是模型對話，而非代理自己的結論。研究流程仍缺少四項要素：

- **存放推導知識的位置。** 代理認為相關的證據、尚未釐清的問題，以及目前的判斷。這些都不是提示詞或回覆。
- **回答前可檢查的內容。** 品質門檻必須讀取結構化的證據說明，而非逐字稿，而且必須在代理仍可採取行動時執行。
- **繼續執行的起點。** 對話紀錄能重播一次執行，卻無法讓執行從中途接續，並使用修改後的設定。
- **可比較的物件。** 兩次執行是兩段對話；比較散文差異並不是測量。兩份 `AgentState` 紀錄則能逐欄位比較。

因此軌跡與狀態是用途不同的兩種產物，本章會同時保留：`RunTrace` 記錄執行從外部看來的樣貌，`AgentState` 記錄代理從內部掌握的資訊。
<!-- ML4T_ZH_END c0005-p000 -->


## The AgentState Schema

`AgentState` in `agent_schemas.py` holds eight fields, in four groups:

- **Identity**: `run_id`, `question`, `cutoff_date` - which run this is, what it is answering,
  and the date past which its evidence may not be published
- **Evidence**: `evidence`, the documents gathered, and `open_questions`, what the agent has
  not resolved yet
- **Provenance**: `tool_trace`, every tool call the run made, including the ones that returned
  nothing
- **Checks**: `quality_gates`, the checks defined below and their outcomes, and
  `synthesis_status`, which moves from `pending` through `in_progress` to `complete`, or to
  `abstained` when the gates refuse the run

It is a plain dataclass with `to_json` and `from_json`, which is all a checkpoint needs to be.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## AgentState 綱要

`AgentState` 位於 `agent_schemas.py`，包含八個欄位，分為四組：

- **識別資訊**：`run_id`、`question`、`cutoff_date`：這是哪次執行、要回答什麼問題，以及證據不得晚於哪個發布日期
- **證據**：`evidence` 是收集到的文件，`open_questions` 是代理尚未釐清的問題
- **資料溯源**：`tool_trace` 記錄這次執行的每次工具呼叫，包括沒有回傳結果的呼叫
- **檢查**：`quality_gates` 記錄下方定義的檢查及其結果；`synthesis_status` 則會從 `pending` 經過 `in_progress` 走向 `complete`，或在品質門檻拒絕此次執行時轉為 `abstained`

它是一個簡單的 dataclass，具有 `to_json` 與 `from_json`；檢查點只需要這些即可。
<!-- ML4T_ZH_END c0006-p000 -->


In [4]:
question = get_demo_question()
state = AgentState(
    question=question.question,
    cutoff_date=question.cutoff_date,
    run_id=RUN_ID,
)
print(f"Run ID:         {state.run_id}")
print(f"Question:       {state.question}")
print(f"Cutoff:         {state.cutoff_date}")
print(f"Status:         {state.synthesis_status}")
print(f"Evidence items: {len(state.evidence)}")
print(f"Quality gates:  {len(state.quality_gates)}")

Run ID:         ch24-state-demo
Question:       Will NVIDIA beat Q4 FY2025 earnings expectations?
Cutoff:         2025-02-20
Status:         pending
Evidence items: 0
Quality gates:  0


## Collecting Evidence

Every search the agent runs appends two records: an evidence item, holding what came back,
and a tool-trace entry, holding that the call happened at all. Keeping both matters, because
a search that returned nothing leaves an evidence list unchanged and a trace one entry longer,
and only the trace can tell "the agent did not look" apart from "the agent looked and found
nothing".

An evidence item carries a `type`, the `source` that produced it, the `timestamp` at which it
was retrieved, the `query` that produced it, and the `content` itself. The three items below
stand in for a short research session and are written out one at a time so each shape is
visible on its own; in a real run they come from the search client of
[`02_tool_contracts`](02_tool_contracts.ipynb).

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 收集證據

代理每執行一次搜尋，就會附加兩筆紀錄：一筆證據項目，保存回傳內容；一筆工具軌跡項目，記錄呼叫確實發生。兩者都必須保存，因為搜尋若沒有結果，證據清單不變，軌跡卻會多一筆；只有軌跡能區分「代理沒有查找」與「代理查找了，但沒有找到」。

一筆證據項目包含 `type`、產生它的 `source`、擷取時的 `timestamp`、產生它的 `query`，以及 `content` 本身。下方三個項目代表一次簡短的研究工作階段，並逐一寫出，讓各自的形式都清楚可見；實際執行時，它們來自 [`02_tool_contracts`](02_tool_contracts.ipynb) 的搜尋用戶端。
<!-- ML4T_ZH_END c0008-p000 -->


The first item is a search on the question itself: what is being said now about the event
being forecast.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
第一個項目直接搜尋問題本身：目前外界如何談論所要預測的事件。
<!-- ML4T_ZH_END c0009-p000 -->


In [5]:
search_evidence: list[dict] = []

search_evidence.append(
    {
        "type": "search_results",
        "source": "web_search",
        "timestamp": as_of.isoformat(),
        "query": "NVIDIA Q4 FY2025 earnings expectations",
        "content": {
            "results": [
                {
                    "title": "NVIDIA suppliers signal continued AI server demand",
                    "url": "https://wsj.com/nvidia-supply-chain",
                    "snippet": "Supply-chain checks point to resilient GPU demand ahead of earnings report.",
                    "published": "2025-02-17",
                },
                {
                    "title": "Analysts raise NVIDIA targets ahead of earnings",
                    "url": "https://nasdaq.com/nvidia-targets",
                    "snippet": "Street revisions remain constructive ahead of the Feb 26 report.",
                    "published": "2025-02-18",
                },
            ]
        },
    }
)

The second is a narrower query on the business line that will decide the outcome. It carries
the same `search_results` type: the type says what kind of evidence this is, not which query
produced it.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
第二個項目針對決定結果的業務部門進行更精確的查詢。它使用相同的 `search_results` 類型：該類型表示證據的種類，而非產生證據的查詢。
<!-- ML4T_ZH_END c0011-p000 -->


In [6]:
search_evidence.append(
    {
        "type": "search_results",
        "source": "web_search",
        "timestamp": as_of.isoformat(),
        "query": "NVIDIA data center revenue growth",
        "content": {
            "results": [
                {
                    "title": "Data center revenue expected to exceed $30B",
                    "url": "https://reuters.com/nvidia-data-center",
                    "snippet": "Consensus estimates point to another record quarter for data center.",
                    "published": "2025-02-15",
                },
            ]
        },
    }
)

The third is a **base rate**: how often the event has happened before, independent of
anything specific to this quarter. It is typed separately because the coverage gate below
requires it, for a reason worth stating plainly. Asked to forecast from current reporting
alone, a model reasons from the narrative in front of it and ignores the frequency; a
historical anchor is what a probability has to be an adjustment away from.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
第三個項目是**基準發生率**：不考慮本季特殊情況時，該事件過去發生的頻率。它被歸為獨立類型，因為下方的覆蓋度門檻要求提供它；理由值得明說。若只依當前報導做預測，模型會根據眼前敘事推理，卻忽略過去頻率；機率判斷必須從歷史基準出發，再作調整。
<!-- ML4T_ZH_END c0013-p000 -->


In [7]:
search_evidence.append(
    {
        "type": "base_rate",
        "source": "web_search",
        "timestamp": as_of.isoformat(),
        "query": "NVIDIA historical earnings beat rate",
        "content": {
            "results": [
                {
                    "title": "NVIDIA has beaten earnings estimates 8 of last 8 quarters",
                    "url": "https://nasdaq.com/nvidia-earnings-history",
                    "snippet": "Strong track record of exceeding consensus, with average surprise of 12%.",
                    "published": "2025-02-10",
                },
            ]
        },
    }
)

Appending them to the state pairs each evidence item with its trace entry.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
將它們附加至狀態時，每筆證據項目都會與對應的軌跡項目配對。
<!-- ML4T_ZH_END c0015-p000 -->


In [8]:
for item in search_evidence:
    state.evidence.append(item)
    state.tool_trace.append({"tool": "search", "query": item["query"], "status": "success"})

print(f"Evidence collected: {len(state.evidence)} items")
print(f"Search calls made: {len(state.tool_trace)}")
for item in state.evidence:
    n_results = len(item["content"].get("results", []))
    print(f'  [{item["type"]}] "{item["query"][:45]}" -> {n_results} results')

Evidence collected: 3 items
Search calls made: 3
  [search_results] "NVIDIA Q4 FY2025 earnings expectations" -> 2 results
  [search_results] "NVIDIA data center revenue growth" -> 1 results
  [base_rate] "NVIDIA historical earnings beat rate" -> 1 results


## Quality Gates

A **quality gate** is a check that runs on the state before the agent is allowed to produce
an answer. It is a contract about evidence: a statement, in code, of what a run must have
gathered for its output to be worth reading. When one fails the agent gathers more or
abstains, and the failure is recorded, so a thin run is visibly thin rather than quietly
confident.

Three gates cover three different ways the evidence can be inadequate:

1. **Coverage** asks whether there is enough evidence, of the required kinds.
2. **Freshness** asks how long ago the agent went and looked.
3. **Consistency** asks whether anything it read was published after the cutoff.

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 品質門檻

**品質門檻**是在允許代理產生答案前，針對狀態執行的檢查。它是一項證據契約：以程式碼說明一次執行必須收集到什麼，輸出才值得閱讀。若某項檢查未通過，代理就繼續收集資訊或棄答，並記錄未通過的情況，讓證據薄弱的執行明顯呈現其不足，而不會悄悄給出自信答案。

三項門檻分別涵蓋證據可能不足的三種情況：

1. **覆蓋度**檢查是否有足夠數量及必要種類的證據。
2. **新近性**檢查代理距離上次查找已經過多久。
3. **一致性**檢查是否讀取了截止日期之後發布的內容。
<!-- ML4T_ZH_END c0017-p000 -->


### Coverage gate

Two conditions. The required types make the gate an argument about *what kind* of evidence a
forecast needs: `search_results` for what is happening now and `base_rate` for how often the
thing has happened before. A model given only current reporting will follow the narrative and
ignore the frequency, which is the classic base-rate neglect, and requiring both types is a
cheap structural defence against it. `min_items` is the separate question of how much: three
records is the chapter's working minimum, low enough that a normal run clears it and high
enough that a single search cannot.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
### 覆蓋度門檻

有兩個條件。必要類型說明預測需要*哪種*證據：`search_results` 對應當前發生的事，`base_rate` 對應該事件過去發生的頻率。若模型只取得當前報導，就會跟著敘事走而忽略發生頻率，這就是典型的基準發生率忽略；要求兩種類型，是成本低廉的結構性防護。`min_items` 則另行回答需要*多少*證據：三筆紀錄是本章採用的最低數量，低到一般執行能達標，高到單次搜尋無法達標。
<!-- ML4T_ZH_END c0018-p000 -->


### Freshness gate

This gate reads the `timestamp` on each evidence item, which is when the agent retrieved the
document, not when the document was published. The two come apart in both directions: a run
that fetched a 2019 paper an hour ago is fresh, and a run that fetched this morning's
reporting three days ago is not. Twenty-four hours is the default window because the
questions in this chapter resolve on news cycles; a question about a quarterly filing would
take a wider one.

A timestamp in the future is treated as a failure rather than as maximal freshness. It means
the clock, the fixture or the checkpoint is wrong, and a gate that reads it as fresh would
pass a run precisely when its record cannot be trusted.

<!-- ML4T_ZH_BEGIN c0019-p000 -->
### 新近性門檻

此門檻讀取每筆證據的 `timestamp`，也就是代理擷取文件的時間，而非文件的發布時間。兩者在兩個方向上都可能不同：一小時前擷取的 2019 年論文仍算新近，三天前擷取的當天早晨報導則不算。預設視窗為 24 小時，因為本章的問題會隨新聞週期得到結果；若問題與季度申報文件有關，就需要更長的視窗。

未來的時間戳記會被視為檢查失敗，而非極高的新近性。這表示時鐘、測試資料或檢查點有誤；若門檻把它當成新近證據，就會恰好在紀錄不可信時放行該次執行。
<!-- ML4T_ZH_END c0019-p000 -->


### Consistency gate

Freshness is about the run; consistency is about the documents. Every search result has to
carry a publication date that parses and that falls before the question's cutoff. The three
ways a result can fail are reported separately, because they mean different things: a missing
date is evidence the tool could not date, an unparseable one is a provider bug, and a
post-cutoff one is hindsight that has already entered the run.

[`02_tool_contracts`](02_tool_contracts.ipynb) filters at retrieval, so in a normal run
nothing reaches here to fail. This gate is the second check on the same property, at the
other end of the pipeline, and it exists because the first one can be bypassed: evidence
arrives from fixtures, from checkpoints, and from tools written after the filter was.

<!-- ML4T_ZH_BEGIN c0020-p000 -->
### 一致性門檻

新近性關乎執行本身；一致性關乎文件。每項搜尋結果都必須附有可解析、且早於問題截止日的發布日期。結果不合格的三種方式會分別回報，因為意義不同：缺少日期表示工具無法確定證據日期；日期無法解析表示供應者有錯誤；日期晚於截止日則表示事後資訊已進入此次執行。

[`02_tool_contracts`](02_tool_contracts.ipynb) 在擷取時篩選，因此正常執行時，不會有結果來到這裡卻未通過檢查。這道門檻位於流程另一端，再次檢查相同屬性；其存在是因為第一道篩選可能被繞過：證據也可能來自測試資料、檢查點，或篩選器建立後才開發的工具。
<!-- ML4T_ZH_END c0020-p000 -->


The three functions live in `agent_schemas.py` beside `AgentState` itself, so the agent in
[`04_research_agent`](04_research_agent.ipynb) checks its evidence against exactly the
definitions demonstrated here. One parser handles the cutoff and every result date, so a date
the cutoff comparison would accept cannot be one a result comparison rejects, and each gate
collects every failure rather than stopping at the first, so one run reports the full extent
of the problem.

<!-- ML4T_ZH_BEGIN c0021-p000 -->
這三個函式位於 `agent_schemas.py`，與 `AgentState` 本身放在一起，因此 [`04_research_agent`](04_research_agent.ipynb) 中的代理會依照這裡示範的相同定義檢查證據。同一個解析器處理截止日和每筆結果的日期，因此截止日比較會接受的日期，不可能在結果比較時遭拒；而且每道門檻都會收集所有失敗項目，不會在第一項失敗時就停止，讓一次執行能回報問題的完整範圍。
<!-- ML4T_ZH_END c0021-p000 -->


### Running the gates

`run_quality_gates` runs the three, stores the outcomes on the state, and returns them, so a
checkpoint carries not only the evidence but the judgement made about it. What to do on a
failure is the caller's decision, not the gate's:
[`04_research_agent`](04_research_agent.ipynb) runs these same three over a finished agent
run and reports what they say about it.

<!-- ML4T_ZH_BEGIN c0022-p000 -->
### 執行門檻檢查

`run_quality_gates` 執行三項檢查，將結果存入狀態並回傳，因此檢查點不只保存證據，也保存對證據的判斷。檢查失敗後該怎麼做，由呼叫端而非門檻決定：[`04_research_agent`](04_research_agent.ipynb) 會對已完成的代理執行套用相同的三項檢查，並報告檢查結果。
<!-- ML4T_ZH_END c0022-p000 -->


In [9]:
gates = run_quality_gates(state, as_of=as_of)

print("Quality gate results:")
for g in gates:
    status = "PASS" if g.passed else "FAIL"
    print(f"  [{status}] {g.gate_name}: {g.reason}")

all_passed = all(g.passed for g in gates)
print(f"\nAll gates passed: {all_passed}")

Quality gate results:
  [PASS] coverage: 3 items covering 2 types
  [PASS] freshness: All evidence within 24h window
  [PASS] consistency: All result dates precede the cutoff

All gates passed: True


All three pass. The run gathered both required evidence types across three records, it
retrieved them at the declared as-of time, and every document it read was published before
the cutoff. Passing gates say the evidence is admissible, and nothing more: none of them has
read a word of what the documents actually claim.

<!-- ML4T_ZH_BEGIN c0024-p000 -->
三項檢查全數通過。這次執行收集的三筆紀錄涵蓋兩種必要證據類型，於宣告的截至時間擷取，且讀取的每份文件都在截止日前發布。門檻通過只表示證據符合納入條件，別無其他含義：這些檢查都沒有閱讀文件實際主張的內容。
<!-- ML4T_ZH_END c0024-p000 -->


## What each gate catches

A gate that has never been seen to fail is a gate nobody should trust. Each state below is
built to breach exactly one contract, so the failure can be attributed to the check that
raised it rather than guessed at.

<!-- ML4T_ZH_BEGIN c0025-p000 -->
## 各道門檻會攔下什麼

從未看過失敗案例的門檻，不應被輕易信任。下方每個狀態都刻意只違反一項契約，讓失敗可歸因於提出它的檢查，而不必靠猜測。
<!-- ML4T_ZH_END c0025-p000 -->


### Too few evidence types

One search, one result, no historical anchor. The coverage gate checks required types before
it checks the count, so this is reported as a missing type.

<!-- ML4T_ZH_BEGIN c0026-p000 -->
### 證據類型太少

一次搜尋、一項結果、沒有歷史基準。覆蓋度門檻會先檢查必要類型，再檢查數量，因此回報缺少類型。
<!-- ML4T_ZH_END c0026-p000 -->


In [10]:
sparse_state = AgentState(question="one search only", cutoff_date="2025-02-20")
sparse_state.evidence.append(
    {
        "type": "search_results",
        "source": "web_search",
        "timestamp": as_of.isoformat(),
        "query": "NVIDIA Q4 FY2025 earnings expectations",
        "content": {"results": [{"title": "Single result", "published": "2025-02-15"}]},
    }
)

sparse_gates = run_quality_gates(sparse_state, as_of=as_of)
print("Gates on the one-search state:")
for g in sparse_gates:
    print(f"  [{'PASS' if g.passed else 'FAIL'}] {g.gate_name}: {g.reason}")

Gates on the one-search state:
  [FAIL] coverage: Missing evidence types: base_rate
  [PASS] freshness: All evidence within 24h window
  [PASS] consistency: All result dates precede the cutoff


### Both types, too little of either

Coverage is two conditions, not one. This state satisfies the type requirement and still
fails, because two records is below the minimum the gate is configured to demand.

<!-- ML4T_ZH_BEGIN c0028-p000 -->
### 兩種類型都有，但各自數量仍不足

覆蓋度有兩個條件，而非一個。這個狀態符合類型要求，卻仍未通過，因為兩筆紀錄低於門檻設定的最低數量。
<!-- ML4T_ZH_END c0028-p000 -->


In [11]:
thin_state = AgentState(question="both types, two records", cutoff_date="2025-02-20")
thin_state.evidence.extend(
    [
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA Q4 FY2025 earnings expectations",
            "content": {"results": [{"title": "Supply chain check", "published": "2025-02-17"}]},
        },
        {
            "type": "base_rate",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA historical earnings beat rate",
            "content": {"results": [{"title": "Beat history", "published": "2025-02-10"}]},
        },
    ]
)

thin_gates = run_quality_gates(thin_state, as_of=as_of)
print("Gates on the two-record state:")
for g in thin_gates:
    print(f"  [{'PASS' if g.passed else 'FAIL'}] {g.gate_name}: {g.reason}")

Gates on the two-record state:
  [FAIL] coverage: Only 2 evidence items (need 3)
  [PASS] freshness: All evidence within 24h window
  [PASS] consistency: All result dates precede the cutoff


### A document published after the cutoff

This is the failure that matters most and shows up least. The state has enough evidence of
both types, retrieved on time, and one of its search results was published on 2025-02-27:
the day after NVIDIA reported. An agent reading it is not forecasting, and its forecast will
look excellent.

<!-- ML4T_ZH_BEGIN c0030-p000 -->
### 截止日後發布的文件

這是最重要、卻最不容易出現的失敗情況。此狀態有足量且涵蓋兩種類型的證據，也在規定時間擷取，但其中一項搜尋結果發布於 2025-02-27：NVIDIA 公布財報的隔天。讀到這份資料的代理並不是在預測，儘管其預測看起來會很出色。
<!-- ML4T_ZH_END c0030-p000 -->


In [12]:
bad_state = AgentState(question="reads past the cutoff", cutoff_date="2025-02-20")
bad_state.evidence.extend(
    [
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA Q4 FY2025 earnings expectations",
            "content": {
                "results": [
                    {"title": "Pre-cutoff article", "published": "2025-02-18"},
                    {"title": "Earnings beat confirmed", "published": "2025-02-27"},
                ]
            },
        },
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA data center revenue growth",
            "content": {"results": [{"title": "Data center outlook", "published": "2025-02-15"}]},
        },
        {
            "type": "base_rate",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA historical earnings beat rate",
            "content": {"results": [{"title": "Base rate data", "published": "2025-02-10"}]},
        },
    ]
)

bad_gates = run_quality_gates(bad_state, as_of=as_of)
print("Gates on the state that read past its cutoff:")
for g in bad_gates:
    print(f"  [{'PASS' if g.passed else 'FAIL'}] {g.gate_name}: {g.reason}")
    for issue in g.details.get("issues", []) if not g.passed else []:
        print(f"    -> {issue}")

Gates on the state that read past its cutoff:
  [PASS] coverage: 3 items covering 2 types
  [PASS] freshness: All evidence within 24h window
  [FAIL] consistency: 1 cutoff violations found
    -> Post-cutoff result: 'Earnings beat confirmed' (published 2025-02-27, cutoff 2025-02-20)


### Stale evidence

Freshness asks a different question from consistency. Consistency asks whether a document
was published before the cutoff; freshness asks how long ago the agent went and looked. A
run that gathered its evidence two days ago and is producing a forecast now has been reading
a stale snapshot of the world, whatever the publication dates say.

The state below carries both required types and dates every document well before the cutoff,
so coverage and consistency pass. Its retrieval timestamps sit 48 hours before the declared
as-of time, which is past the gate's 24-hour default.

<!-- ML4T_ZH_BEGIN c0032-p000 -->
### 過時的證據

新近性與一致性所問的問題不同。一致性問的是文件是否在截止日前發布；新近性問的是代理多久以前進行查找。若一次執行在兩天前收集證據，現在才產生預測，那麼無論發布日期如何，它讀到的都是世界的過時快照。

下方狀態具備兩種必要類型，所有文件日期也都遠早於截止日，所以覆蓋度與一致性檢查會通過。但其擷取時間戳記比宣告的截至時間早 48 小時，超過門檻預設的 24 小時。
<!-- ML4T_ZH_END c0032-p000 -->


In [13]:
stale_ts = (as_of - timedelta(hours=48)).isoformat()
stale_state = AgentState(question="gathered two days ago", cutoff_date="2025-02-20")
stale_state.evidence.extend(
    [
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": stale_ts,
            "query": "NVIDIA earnings (stale)",
            "content": {"results": [{"title": "Old article", "published": "2025-02-12"}]},
        },
        {
            "type": "base_rate",
            "source": "web_search",
            "timestamp": stale_ts,
            "query": "NVIDIA beat rate (stale)",
            "content": {"results": [{"title": "Historical beats", "published": "2025-02-10"}]},
        },
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": stale_ts,
            "query": "NVIDIA guidance (stale)",
            "content": {"results": [{"title": "Guidance recap", "published": "2025-02-11"}]},
        },
    ]
)
stale_gates = run_quality_gates(stale_state, as_of=as_of)
print("Stale state gates:")
for g in stale_gates:
    status = "PASS" if g.passed else "FAIL"
    print(f"  [{status}] {g.gate_name}: {g.reason}")

Stale state gates:
  [PASS] coverage: 3 items covering 2 types
  [FAIL] freshness: 3 evidence items exceed 24h age limit
  [PASS] consistency: All result dates precede the cutoff


## Checkpointing

`AgentState` is a dataclass of plain lists and strings, so serialising it is
`json.dumps(asdict(self))` and restoring it is the constructor. That simplicity is the point:
a checkpoint format that needs a custom encoder is a format that will silently drop a field
the day someone adds one.

The record is JSON rather than a pickle for the same reason it lives outside the context
window. A pickle needs this codebase at this version to be read at all; a JSON checkpoint can
be opened in a text editor, diffed against another run, queried without loading Python, and
read in five years by something that has never heard of `AgentState`.

<!-- ML4T_ZH_BEGIN c0034-p000 -->
## 建立檢查點

`AgentState` 是由普通清單與字串組成的 dataclass，因此序列化只需 `json.dumps(asdict(self))`，還原則只需呼叫建構子。簡單正是重點：若檢查點格式需要自訂編碼器，日後有人新增欄位時，就可能悄悄遺漏該欄位。

紀錄採用 JSON，而非 pickle，理由與它存放在上下文視窗之外相同。要讀取 pickle，必須有此版本的程式碼；JSON 檢查點則可在文字編輯器中開啟、與另一次執行比較、無須載入 Python 即可查詢，五年後也能由從未聽過 `AgentState` 的程式讀取。
<!-- ML4T_ZH_END c0034-p000 -->


In [14]:
checkpoint = state.to_json()
print(f"Checkpoint size: {len(checkpoint):,} bytes")

data = json.loads(checkpoint)
print(f"Top-level fields: {list(data.keys())}")
print(f"Evidence items:   {len(data['evidence'])}")
print(f"Quality gates:    {len(data['quality_gates'])}")

Checkpoint size: 3,166 bytes
Top-level fields: ['question', 'cutoff_date', 'run_id', 'evidence', 'open_questions', 'tool_trace', 'quality_gates', 'synthesis_status']
Evidence items:   3
Quality gates:    3


Restoring it has to give back the same record, not merely a similar one. Comparing the two
serialised forms checks every field at once, including the ones a hand-written comparison
would forget to look at.

<!-- ML4T_ZH_BEGIN c0036-p000 -->
還原後必須得到完全相同的紀錄，而不只是相似的紀錄。比較兩份序列化形式，就能一次檢查所有欄位，包括人工逐項比較時可能忘記看的欄位。
<!-- ML4T_ZH_END c0036-p000 -->


In [15]:
restored = AgentState.from_json(checkpoint)
print(f"Restored run ID:   {restored.run_id}")
print(f"Restored evidence: {len(restored.evidence)} items")
print(f"Restored gates:    {len(restored.quality_gates)} results")

assert restored.to_json() == checkpoint, "checkpoint did not survive the round trip"
print("Round trip: byte-identical")

Restored run ID:   ch24-state-demo
Restored evidence: 3 items
Restored gates:    3 results
Round trip: byte-identical


## Replay: what happens without the historical anchor

The reason to checkpoint is not only to resume. A saved state is also the input to an
experiment: restore it, change one thing, re-run the gates, and read what moved. The
question below is whether the base-rate evidence is load-bearing or decorative.

<!-- ML4T_ZH_BEGIN c0038-p000 -->
## 重播：沒有歷史基準會如何

建立檢查點不只是為了接續執行。已儲存的狀態也能作為實驗輸入：還原狀態、改變一件事、重新執行門檻檢查，再觀察哪些結果改變。下方要問的是，基準發生率證據究竟是不可或缺，還是僅作點綴。
<!-- ML4T_ZH_END c0038-p000 -->


In [16]:
print("=== Original run ===")
for g in state.quality_gates:
    print(f"  [{('PASS' if g.passed else 'FAIL')}] {g.gate_name}")

ablation = AgentState.from_json(checkpoint)
ablation.evidence = [e for e in ablation.evidence if e["type"] != "base_rate"]
ablation_gates = run_quality_gates(ablation, as_of=as_of)

print("\n=== Without base-rate evidence ===")
print(f"Evidence items: {len(ablation.evidence)} (was {len(state.evidence)})")
for g in ablation_gates:
    print(f"  [{('PASS' if g.passed else 'FAIL')}] {g.gate_name}")

=== Original run ===
  [PASS] coverage
  [PASS] freshness
  [PASS] consistency

=== Without base-rate evidence ===
Evidence items: 2 (was 3)
  [FAIL] coverage
  [PASS] freshness
  [PASS] consistency


In [17]:
if not all(g.passed for g in ablation_gates):
    ablation.synthesis_status = "abstained"
    print(f"Synthesis status: {ablation.synthesis_status}")
    print("No forecast is produced: the coverage contract is not satisfied.")
else:
    print("All gates still pass; the base rate was not required for coverage.")

Synthesis status: abstained
No forecast is produced: the coverage contract is not satisfied.


Dropping the base rate takes the coverage gate below its required types, and the run
abstains. That is the behaviour worth arguing about rather than the code: whether a
forecast should be refused because no historical frequency was found is a research
decision, and writing it as a gate is what makes it a decision someone can point at
rather than an accident of what the agent happened to search for.

<!-- ML4T_ZH_BEGIN c0041-p000 -->
移除基準發生率後，覆蓋度門檻就缺少必要類型，這次執行會棄答。值得討論的是這種行為，而非程式碼本身：找不到歷史頻率時是否應拒絕預測，是研究決策。把它寫成門檻，才讓這項決策可以被明確指出，而非代理碰巧搜尋了什麼所造成的意外。
<!-- ML4T_ZH_END c0041-p000 -->


## Where this sits in the memory hierarchy

Three kinds of memory are usually distinguished, and they are easy to conflate because a
language model presents all three as text in one prompt. **Working memory** is the context
window: what the model can see this turn, built in
[`01_react_reasoning`](01_react_reasoning.ipynb). **Short-term memory** is the run's own
durable record, which is the `AgentState` above: it outlives the conversation and can be
read by something other than the model. **Long-term memory** is knowledge carried across
runs, retrieved on demand rather than held in the prompt, which is what Chapter 22 builds
with retrieval-augmented generation.

<!-- ML4T_ZH_BEGIN c0042-p000 -->
## 這在記憶層級中的位置

通常會區分三種記憶，但語言模型會將三者都呈現為同一份提示詞中的文字，因此很容易混淆。**工作記憶**是上下文視窗：模型在這一輪能看到的內容，建構於 [`01_react_reasoning`](01_react_reasoning.ipynb)。**短期記憶**是這次執行自身的持久紀錄，也就是上方的 `AgentState`：它在對話結束後仍存在，也能由模型之外的程式讀取。**長期記憶**是跨執行保留的知識，按需擷取而非長期放在提示詞中；第 22 章以檢索增強生成建立的就是這種記憶。
<!-- ML4T_ZH_END c0042-p000 -->


## Key Takeaways

1. **State the agent can be audited on has to live outside the context window.** A message
   history is a transcript, not a record: it cannot be queried, diffed, or replayed, and it is
   gone when the conversation ends.
2. **A gate is a contract about evidence, checked before synthesis.** It says what a run must
   have gathered to be allowed to produce an answer, in code, so a thin run abstains instead
   of guessing confidently.
3. **Separate when evidence was retrieved from when it was published.** The two answer
   different questions - is this run stale, and did this run read the future - and conflating
   them lets one hide the other.
4. **Abstention is an outcome, not a failure.** A forecast the evidence does not support costs
   more than no forecast, because it is scored as if it were a judgement.
5. **A checkpoint turns an ablation into a one-line experiment.** Restore, drop a class of
   evidence, re-run the gates, and read what changed.

**Known limitations of what is built here.** The gates check that evidence exists, is recent,
and predates the cutoff; none of them looks at whether it is any good, whether five results
are five sources or one wire story copied five times, or whether the sources contradict each
other. Freshness is measured against a declared as-of rather than wall-clock time, which is
what makes replay possible and also means a stale run replays as fresh. And the gates run
once before synthesis, so evidence that arrives during synthesis is ungated.

**Next**: [`04_research_agent`](04_research_agent.ipynb) combines the provider, the search
tool, and this state into an agent that gathers evidence, runs these gates, and either
forecasts or abstains.


<!-- ML4T_ZH_BEGIN c0043-p000 -->
## 重點整理

1. **可供稽核的代理狀態必須存放在上下文視窗之外。** 訊息歷史是逐字稿，而非狀態紀錄：它無法供查詢、比較差異或重播，對話結束後也就消失。
2. **門檻是合成答案前檢查的證據契約。** 它以程式碼規定一次執行必須收集什麼才能產生答案，讓證據不足的執行棄答，而非自信地猜測。
3. **區分證據擷取時間與發布時間。** 兩者分別回答不同問題：這次執行是否過時？是否讀到了未來資訊？混為一談會讓其中一個問題掩蓋另一個。
4. **棄答是一種結果，而非失敗。** 缺乏證據支持的預測，代價比不預測更高，因為它會被當成實際判斷來評分。
5. **檢查點讓消融分析變成簡單的實驗。** 還原狀態、移除一類證據、重新執行門檻，再查看變化。

**此處實作的已知限制。** 門檻會檢查證據是否存在、是否新近，以及是否早於截止日；但不會檢查證據品質、五項結果究竟來自五個來源還是同一篇被轉載五次的通訊社報導，也不會檢查來源是否互相矛盾。新近性依宣告的截至時間而非目前時鐘衡量，因此可以重播，但過時的執行在重播時也可能顯得新近。此外，門檻只在合成答案前執行一次，所以合成過程中新取得的證據不會經過門檻檢查。

**下一步**：[`04_research_agent`](04_research_agent.ipynb) 將供應者、搜尋工具與此狀態結合為代理；代理會收集證據、執行這些門檻，然後預測或棄答。
<!-- ML4T_ZH_END c0043-p000 -->

**Book**: Section 24.3 covers the memory hierarchy in depth, including vector stores
and RAG-backed long-term memory.

<!-- ML4T_ZH_BEGIN c0043-p001 -->
**書籍**：第 24.3 節深入介紹記憶層級，包括向量儲存庫，以及由 RAG 支援的長期記憶。
<!-- ML4T_ZH_END c0043-p001 -->
